# 03 · One-stage anchor-free: YOLOv8 (Ultralytics)
YOLOv8 is anchor-free: each grid cell predicts distances to the 4 box sides (DFL), positives are chosen by the
task-aligned assigner (TOOD). Same data copy, frame step and evaluator as 01/02.
This notebook also writes the **first leaderboard submission**.

Kaggle: Accelerator **GPU**, **Internet on**. Run with *Save Version → Save & Run All*.

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
sys.path.insert(0, next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists()))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.208', 'pycocotools'], check=True)

import pickle
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from common import *

seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2') if not FAST_DEV else Path('work_dev')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)
WEIGHTS = Path('weights'); WEIGHTS.mkdir(exist_ok=True)   # best.pt files are kept as notebook output
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Protocol

In [ ]:
STEP = 5
EPOCHS = 15 if not FAST_DEV else 1
yaml_path = prepare_training_copy(ROOT, TMP / 'data', step_train=STEP, step_val=STEP)
val_frames = list_frames(ROOT / 'val', STEP)
evaluator = CocoEvaluator(ROOT, val_frames)
print('val frames:', len(val_frames))

COMMON_ARGS = dict(data=str(yaml_path), epochs=EPOCHS, seed=SEED, deterministic=True, cos_lr=True, close_mosaic=3,
                   workers=4, project=str((TMP / 'runs').resolve()), exist_ok=True, plots=True, verbose=False,
                   device=0 if torch.cuda.is_available() else 'cpu')

## 2. Experiments
Search range: input resolution {640, 1024}; extra high-resolution head {no, P2 (stride 4)}.
* `yolo_640` – YOLOv8s, imgsz 640 (baseline).
* `yolo_1024` – YOLOv8s, imgsz 1024 (objects ~1.6× larger in pixels after resizing).
* `yolo_p2_1024` – YOLOv8s + P2 head (stride-4 detection level for tiny objects), imgsz 1024.
  Weights for the shared layers come from `yolov8s.pt`; the new P2 layers start from scratch.

In [ ]:
EXPERIMENTS = {
    'yolo_640':     dict(cfg='yolov8s.pt', imgsz=640, batch=16),
    'yolo_1024':    dict(cfg='yolov8s.pt', imgsz=1024, batch=8),
    'yolo_p2_1024': dict(cfg='yolov8s-p2.yaml', load='yolov8s.pt', imgsz=1024, batch=8),
}
RUN = list(EXPERIMENTS)
if FAST_DEV:
    EXPERIMENTS = {k: dict(v, imgsz=320, batch=4, cfg=v['cfg'].replace('8s', '8n'), load=v.get('load', '').replace('8s', '8n') or None)
                   for k, v in EXPERIMENTS.items()}
    RUN = ['yolo_640', 'yolo_p2_1024']


def build_yolo(cfg):
    model = YOLO(cfg['cfg'])
    if cfg.get('load'):
        model.load(cfg['load'])  # transfer all layers with matching shapes
    return model

## 3. Train and evaluate (common COCO evaluator on raw val frames)

In [ ]:
results = {}
for name in RUN:
    cfg = EXPERIMENTS[name]
    print(f'\n===== {name} =====')
    seed_everything()
    model = build_yolo(cfg)
    model.train(name=name, imgsz=cfg['imgsz'], batch=cfg['batch'], **COMMON_ARGS)
    run_dir = Path(model.trainer.save_dir)
    best = run_dir / 'weights' / 'best.pt'
    shutil.copy(best, WEIGHTS / f'{name}.pt')
    shutil.copy(run_dir / 'results.csv', OUT / f'ultra_{name}.csv')  # per-epoch curves
    model = YOLO(best)
    preds, fps = predict_ultralytics(model, val_frames, imgsz=cfg['imgsz'])
    m = evaluator.evaluate(preds)
    results[name] = preds
    log_result(dict(model=name, family='one-stage anchor-free', epochs=EPOCHS, step=STEP, imgsz=cfg['imgsz'], fps=fps, **m),
               OUT / 'metrics.csv')
    pickle.dump(preds, open(TMP / f'val_preds_{name}.pkl', 'wb'))
    print(name, {k: m[k] for k in ['mAP', 'AP50', 'AP_small']}, 'fps', fps)
    del model; torch.cuda.empty_cache()

## 4. Results, broken down by object size

In [ ]:
tab = pd.read_csv(OUT / 'metrics.csv')
cols = ['model', 'family', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AR500', 'fps']
print(tab[cols].to_string(index=False))

size_tabs = []
for name, preds in results.items():
    t = evaluator.evaluate_by_size(preds); t.insert(0, 'model', name); size_tabs.append(t)
size_tab = pd.concat(size_tabs)
size_tab.to_csv(OUT / 'size_breakdown_yolo.csv', index=False)
print(size_tab.pivot(index='size', columns='model', values='AP50'))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for name in results:
    h = pd.read_csv(OUT / f'ultra_{name}.csv'); h.columns = h.columns.str.strip()
    ax[0].plot(h.epoch, h['metrics/mAP50-95(B)'], label=name)
    s = size_tab[(size_tab.model == name) & (size_tab['size'] != 'all')]
    ax[1].plot(s['size'], s.AP50, marker='o', label=name)
ax[0].set_title('val mAP50-95 per epoch (Ultralytics metric)'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].set_title('AP50 by object size (sqrt area, px)'); ax[1].legend()
plt.tight_layout(); plt.savefig(FIG / 'yolo_curves_sizes.png', dpi=120); plt.show()

## 5. Failure cases of the best YOLO run

In [ ]:
BEST = max(results, key=lambda n: evaluator.evaluate(results[n])['mAP'])
print('best:', BEST)
err = frame_errors(ROOT, val_frames, results[BEST], score_thr=0.3)
print(err[['missed', 'fp', 'wrong_cls']].sum())
r = err.sort_values('missed').iloc[-1]
show_frame(ROOT, r['path'], r['clip'], results[BEST], 0.3, title=f'{BEST}: most missed {r.key} (GT {r.n_gt}, missed {r.missed})')
plt.savefig(FIG / 'yolo_failure.png', dpi=90, bbox_inches='tight'); plt.show()

## 6. First submission (best YOLO run, plain inference)

In [ ]:
model = YOLO(WEIGHTS / f'{BEST}.pt')
test_preds, fps = predict_ultralytics(model, test_frames(ROOT), imgsz=EXPERIMENTS[BEST]['imgsz'])
write_submission(test_preds, ROOT, f'submission_{BEST}.csv')
print('test fps:', fps)

## 7. Notes for the report
*(your analysis: how locations are assigned without anchors (TAL, stride / centre sampling), why tiny & dense
objects suffer, effect of resolution and of the P2 head, comparison with the tuned anchor-based models by size)*